# T2.5 파일럿 베이스라인 1시드 학습 + 진단 4종 (Colab T4)

역할: **브랜치 가져오기 → 데이터 복사·해제 → `scripts/train_pilot_baseline.py` → `scripts/run_pilot_diagnosis.py` → `results/pilot/` 커밋·push**.
로직은 `src/sr_pipeline/`에 있고 이 노트북에는 넣지 않습니다.

- 이 학습은 **진단용 파일럿**입니다. 결과 표·집계에 넣지 않고 `results/pilot/`에만 둡니다.
- 에폭·에폭당 패치·배치·학습률·옵티마이저·스케줄·증강은 `pilot_determined`가 `null`인 동안 코드에 기본값이 없습니다. 셀 1에서 정해 넣으세요(`None`이면 중단).
- **셀을 위에서부터 하나씩 실행**하고 각 셀이 성공했는지 확인하세요("모두 실행"은 쓰지 않습니다).
- 같은 런타임에서 다시 실행하면 셀 2가 저장소를 최신으로 갱신하고, 셀 3-1이 이전 SR 결과(`SR_DIR`)를 지웁니다(진단 산출물은 셀 5가 실행 시 이전 진단 파일만 교체합니다).
- 결과는 **본인 이름 브랜치**에 push합니다. main은 PR로만 바뀝니다(AGENTS.md).
- 체크포인트(`*.pth`)는 Drive에 두고 커밋하지 않습니다. 저장소가 Public이라 링크·토큰·경로를 출력이나 커밋에 남기지 않습니다. 실행 후 셀 출력은 지우고 커밋합니다.

In [ ]:
# 1. 실행 설정
REPO_URL = "https://github.com/minyi-k03/VideoProcessing-Project.git"
BRANCH = "YongMin"
SEED = 0
EPOCHS = 30                    # 파일럿 에폭 수(팀 결정). 1에폭은 작동 확인용 시험 실행이었다
PATCHES_PER_EPOCH = 10000
BATCH_SIZE = 16
LEARNING_RATE = 1e-3
OPTIMIZER = "adam"
SCHEDULER = "none"
AUGMENTATION = "rot90x4_hflip"
EDGE_TOP_Q = 20.0
ARCHIVE_NAME = "div2k_x4_p96_n10000_s42_e8bc539e.zip"   # Drive의 파일럿 풀 압축 파일 이름(T1.5). 경로는 셀 3이 찾는다
CKPT_NAME = "pilot_baseline_s0.pth"                      # 체크포인트 파일 이름. zip과 같은 Drive 폴더에 저장하고 커밋하지 않는다
DATA_DIR = "/content/data"
SR_DIR = "/content/pilot_sr"
missing = [k for k, v in dict(BRANCH=BRANCH, EPOCHS=EPOCHS, PATCHES_PER_EPOCH=PATCHES_PER_EPOCH, BATCH_SIZE=BATCH_SIZE,
           LEARNING_RATE=LEARNING_RATE, OPTIMIZER=OPTIMIZER, SCHEDULER=SCHEDULER, AUGMENTATION=AUGMENTATION,
           EDGE_TOP_Q=EDGE_TOP_Q).items() if v is None]
assert not missing, f"값을 채우세요: {missing}"
assert BRANCH != "main", "결과는 본인 이름 브랜치에 push한다. main은 PR로만 바뀐다(AGENTS.md)."


In [ ]:
# 2. 브랜치 가져오기 (이미 받아 둔 런타임이면 최신으로 갱신한다)
import os, subprocess
REPO_DIR = "/content/VideoProcessing-Project"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
%cd /content/VideoProcessing-Project
!pip install -q -r requirements.txt

In [ ]:
# 3. 데이터 준비: Drive 마운트 → zip을 이름으로 찾기 → 런타임 로컬로 복사·해제 (SPEC 실행 창구, ADR-0005)
# 어느 단계든 실패하면 그 자리에서 멈춘다. 경로는 출력하지 않는다(저장소가 Public).
import glob, os, shutil, zipfile
from google.colab import drive
drive.mount("/content/drive")

matches = glob.glob(f"/content/drive/MyDrive/**/{ARCHIVE_NAME}", recursive=True)
assert len(matches) == 1, (
    f"MyDrive 아래에서 {ARCHIVE_NAME}을 {len(matches)}개 찾았습니다(1개여야 함). "
    "0개면 공유 문서함 폴더일 수 있으니 '내 드라이브에 바로가기 추가'를 한 뒤 다시 실행하세요.")
DATA_ZIP = matches[0]
CKPT_PATH = os.path.join(os.path.dirname(DATA_ZIP), CKPT_NAME)   # 셀 4가 쓴다

os.makedirs(DATA_DIR, exist_ok=True)
shutil.copy(DATA_ZIP, "/content/pilot_data.zip")
with zipfile.ZipFile("/content/pilot_data.zip") as archive:
    archive.extractall(DATA_DIR)
for part in ("patch_pool", "testsets"):
    assert os.path.isdir(os.path.join(DATA_DIR, part)), f"압축 해제 결과에 {part}/ 폴더가 없습니다."
print("데이터 준비 완료:", sorted(os.listdir(os.path.join(DATA_DIR, "testsets"))))


In [ ]:
# 3-1. 재실행 준비: 이전 실행의 SR 결과 폴더를 지운다 (처음 실행이면 지울 것이 없다)
# - 학습 스크립트는 --sr-dir가 이미 있으면 거부한다(이전 SR 결과와 섞이지 않게).
import shutil

shutil.rmtree(SR_DIR, ignore_errors=True)
print("SR_DIR 정리 완료")


In [ ]:
# 4. 학습 + SR 결과 생성 (T4). 학습 전에 테스트셋·SR 폴더를 먼저 확인한다.
# 학습 설정과 실행 시점 커밋(git rev-parse HEAD)은 results/pilot/pilot_train_settings.json에 기록된다.
!PYTHONPATH=src python scripts/train_pilot_baseline.py --data-dir "{DATA_DIR}" --checkpoint-path "{CKPT_PATH}" --sr-dir "{SR_DIR}" --output-dir results/pilot --seed {SEED} --epochs {EPOCHS} --patches-per-epoch {PATCHES_PER_EPOCH} --batch-size {BATCH_SIZE} --learning-rate {LEARNING_RATE} --optimizer "{OPTIMIZER}" --scheduler "{SCHEDULER}" --augmentation "{AUGMENTATION}"

In [ ]:
# 5. 진단 4종 (오차 지도, 에지·평탄 PSNR(Y), 주파수 스펙트럼, YCbCr 채널별 오차 + 최악 영상)
!PYTHONPATH=src python scripts/run_pilot_diagnosis.py --sr-dir "{SR_DIR}" --hr-dir "{DATA_DIR}/testsets" --output-dir results/pilot --edge-top-q {EDGE_TOP_Q}

In [ ]:
# 6. results/pilot/ 커밋·push (ADR-0005, 노트북 01 셀 3-1과 같은 방식)
# - 토큰은 getpass로만 받고 명령행·remote URL·파일·셀 출력에 남기지 않는다(GIT_ASKPASS 임시 스크립트로 전달).
# - 커밋 작성자 이름·이메일은 이 런타임의 명령에만 쓰고 출력하지 않는다.
# - git 명령이 하나라도 실패하면 멈춘다(커밋이 실패했는데 push가 성공처럼 보이는 것을 막는다).
# - 실행 후 이 노트북의 셀 출력은 지우고 커밋한다.
import getpass, os, stat, subprocess, tempfile

for required in ("results/pilot/pilot_train_settings.json", "results/pilot/per_image_diagnosis.csv"):
    assert os.path.isfile(required), f"{required}가 없습니다. 셀 4·5가 끝까지 성공했는지 확인하세요(커밋하지 않고 멈춤)."

GH_USER = input("GitHub 사용자명: ").strip()
GIT_NAME = input("커밋 작성자 이름 (이 런타임에서만 사용): ").strip()
GIT_EMAIL = input("커밋 작성자 이메일 (이 런타임에서만 사용): ").strip()
token = getpass.getpass("GitHub 토큰 (저장되지 않음): ")

askpass = tempfile.NamedTemporaryFile("w", suffix=".sh", delete=False)
askpass.write('#!/bin/sh\ncase "$1" in\n  Username*) echo "$GH_USER";;\n  *) echo "$GH_TOKEN";;\nesac\n')
askpass.close()
os.chmod(askpass.name, stat.S_IRWXU)

env = {**os.environ, "GIT_ASKPASS": askpass.name, "GIT_TERMINAL_PROMPT": "0",
       "GH_USER": GH_USER, "GH_TOKEN": token}
ident = ["-c", f"user.name={GIT_NAME}", "-c", f"user.email={GIT_EMAIL}"]


def git(*args):
    shown = " ".join(a for a in args if not a.startswith("user."))  # 이름·이메일은 출력하지 않는다
    r = subprocess.run(["git", *args], env=env, capture_output=True, text=True)
    print(f"$ git {shown}  -> exit {r.returncode}")
    if r.stdout.strip():
        print(r.stdout.strip())
    if r.stderr.strip():
        print(r.stderr.strip())
    if r.returncode != 0:
        raise RuntimeError(f"git {shown} 실패")
    return r


try:
    git("add", "results/pilot")
    git("status", "--short", "results/pilot")
    git(*ident, "commit", "-m", f"results: T2.5 pilot baseline seed {SEED}, {EPOCHS} epochs diagnosis")
    git("push", "origin", f"HEAD:{BRANCH}")
    print("✅ push 완료")
finally:
    os.remove(askpass.name)
    del token, env
